# Module B Optimization Report (SubTask 4 and 5)

This notebook summarizes the indexing strategy, benchmark setup, and measured impact for Module B.

## 1) Endpoints and Query Patterns

Hot endpoints selected for optimization:
- `GET /posts`
- `GET /posts/{post_id}/comments`

Target query clauses:
- Posts feed: `WHERE p.IsActive = TRUE ORDER BY p.PostDate DESC`
- Comments list: `WHERE c.PostID = ? AND c.IsActive = TRUE ORDER BY c.CommentDate ASC`

## 2) Indexing Strategy (SubTask 4)

Implemented targeted indexes:
1. `idx_post_active_postdate ON Post(IsActive, PostDate DESC)`
2. `idx_comment_post_active_date ON Comment(PostID, IsActive, CommentDate ASC)`

Baseline FK-support indexes are retained in both benchmark stages.

In [ ]:
import json
from pathlib import Path

results_path = Path('index_benchmark_results.json')
with results_path.open('r', encoding='utf-8') as f:
    results = json.load(f)

results['benchmark_params']

In [ ]:
before = results['stages'][0]
after = results['stages'][1]
speedup = results['speedup']

summary = {
    'before_sql_posts_avg_ms': before['sql_ms']['list_posts']['avg_ms'],
    'after_sql_posts_avg_ms': after['sql_ms']['list_posts']['avg_ms'],
    'before_sql_comments_avg_ms': before['sql_ms']['list_comments']['avg_ms'],
    'after_sql_comments_avg_ms': after['sql_ms']['list_comments']['avg_ms'],
    'before_api_posts_avg_ms': before['api_ms']['list_posts']['avg_ms'],
    'after_api_posts_avg_ms': after['api_ms']['list_posts']['avg_ms'],
    'before_api_comments_avg_ms': before['api_ms']['list_comments']['avg_ms'],
    'after_api_comments_avg_ms': after['api_ms']['list_comments']['avg_ms'],
    'posts_sql_speedup': speedup['posts_sql_speedup'],
    'comments_sql_speedup': speedup['comments_sql_speedup'],
    'posts_api_speedup': speedup['posts_api_speedup'],
    'comments_api_speedup': speedup['comments_api_speedup'],
}
summary

In [ ]:
print('Before EXPLAIN (comments):')
for row in before['explain']['list_comments']:
    print(row)

print('\nAfter EXPLAIN (comments):')
for row in after['explain']['list_comments']:
    print(row)

## 3) Benchmark Integrity Checks (SubTask 5)

- Same benchmark parameters are used for both stages (`iterations`, `warmup`, `limit`, `offset`, `comment_post_id`).
- Both API response times and SQL query times are recorded before and after indexing.
- EXPLAIN output is captured side-by-side with `key`, `type`, `rows`, and `extra`.

## 4) Conclusion

- The comments endpoint shows a clear plan shift and measurable speedup after indexing.
- The posts endpoint index is included as a targeted strategy for its filter/sort pattern and can be re-evaluated with larger or differently distributed production data.